# Evaluation: Antwort gegen die Top-K-Chunks des Retrievers

Dieses Notebook durchlaeuft die **gesamte Pipeline** ueber das gespeicherte Testset und bewertet die finale Antwort gegen die **Top-K (=5) tatsaechlich gelieferten Chunks** aus der gesamten Wissensbasis.

- `score_mean`: mittlere Kosinus-Aehnlichkeit zwischen Antwort-Embedding und den Top-K-Chunk-Embeddings.
- `score_max`: maximale Kosinus-Aehnlichkeit zwischen Antwort-Embedding und einem Top-K-Chunk.

Das Testset wird einmalig mit `create_testset.py` erzeugt und hier nur geladen.

## 0. Setup, Factory und Parameter

In [1]:
import sys
import json
import time
import datetime
import statistics
import importlib.util
from pathlib import Path

# Projekt-Root finden und VOR den App-Importen auf sys.path legen, damit das
# Notebook unabhaengig vom Startverzeichnis von Jupyter laeuft.
ARBEITSVERZEICHNIS = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        pfad
        for pfad in (ARBEITSVERZEICHNIS, *ARBEITSVERZEICHNIS.parents)
        if (pfad / "config.yaml").exists() and (pfad / "app").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Projekt-Root (mit config.yaml und app/) nicht gefunden.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display, HTML
from app.core.config import load_config
from app.core.factory import build_components
from test_verzahnung.rag_evaluation.evaluation_shared import (
    generiere_antwort,
    kosinus_aehnlichkeit,
    kuerzen,
    lade_testset,
    restbudget_ok,
    rufe_retriever_auf,
)

NOTEBOOK_START = time.monotonic()

# ============================================================
# PARAMETER – hier Modelle und Einstellungen für den Lauf wählen
# ============================================================
ANSWER_MODEL = None          # None = Modell aus config.yaml, sonst z. B. "llama3.1:8b"
OUTPUT_FORMAT = "standard"   # Ausgabeformat des Antwortgenerators
TOP_K = 5                    # Anzahl der bewerteten Top-Chunks aus dem Retriever
MAX_CASES = None             # None = gesamtes Testset, sonst Begrenzung der Fallzahl
LLM_TIMEOUT_SECONDS = 90     # Timeout pro LLM-Aufruf
ANSWER_MAX_TOKENS = 450      # Tokenbudget der Antwort
MAX_REVISIONS = 0            # Revisionsrunden des Reviewers (0 = aus, für kürzere Laufzeit)
MAX_RUNTIME_SECONDS = 30 * 60  # Sicherheitsbudget: danach keine neuen Fälle mehr

# ------------------------------------------------------------
EVAL_DIR = PROJECT_ROOT / "test_verzahnung" / "rag_evaluation"
STEP_FILE_PATH = PROJECT_ROOT / "test_verzahnung" / "Input_Daten" / "evaluation_gear.step"
TESTSET_PATH = EVAL_DIR / "rag_evaluation_testset.json"
LOG_DIR = EVAL_DIR / "logs"

config_original = load_config(PROJECT_ROOT / "config.yaml")

# Laufzeitbegrenzte Kopie: gleiche Implementierungen, aber Timeout/Tokenbudget/Revisionen gedeckelt
# und optional ein anderes Antwortmodell.
answer_update = {
    "timeout_s": LLM_TIMEOUT_SECONDS,
    "max_tokens": ANSWER_MAX_TOKENS,
    "max_revisions": MAX_REVISIONS,
}
if ANSWER_MODEL:
    answer_update["model_name"] = ANSWER_MODEL
config = config_original.model_copy(
    update={"answer_generator": config_original.answer_generator.model_copy(update=answer_update)}
)

# CAD-Adapter nur dann auf synthetic_json zurückfallen lassen, wenn OCC/STEP fehlt.
cad_fallback_grund = None
if config.cad_adapter.implementation == "cad_processor_local":
    if importlib.util.find_spec("OCC") is None:
        cad_fallback_grund = f"pythonocc-core/OCC fehlt im Kernel {sys.executable}."
    elif not STEP_FILE_PATH.exists():
        cad_fallback_grund = f"STEP-Datei fehlt: {STEP_FILE_PATH}"
if cad_fallback_grund:
    config = config.model_copy(
        update={"cad_adapter": config.cad_adapter.model_copy(update={"implementation": "synthetic_json"})}
    )
    print(f"WARNUNG: {cad_fallback_grund}")
    print("Fuer diesen Lauf wird nur der CAD-Adapter auf synthetic_json umgestellt.")

print("Baue die Factory-Pipeline auf ...")
components = build_components(config, base_dir=PROJECT_ROOT)
embedder = components.embedder
retriever = components.retriever
answer_gen = components.answer_generator
cad_adapter = components.cad_adapter
synthetic_cad_adapter = components.synthetic_cad_adapter

print("\nKonfiguration:")
print(f"  Antwortmodell:     {config.answer_generator.model_name}")
print(f"  Antwortstrategie:  {config.answer_generator.implementation}")
print(f"  Review aktiv:      {config.answer_generator.enable_review}")
print(f"  Revisionsrunden:   {config.answer_generator.max_revisions}")
print(f"  Top-K Chunks:      {TOP_K}")
print(f"  Output-Format:     {OUTPUT_FORMAT}")
print(f"  Laufzeitbudget:    {MAX_RUNTIME_SECONDS / 60:.0f} Minuten")
print(f"  Collection:        {config.vector_store.collection_name}")
print(f"  Testset:           {TESTSET_PATH}")


WARNUNG: pythonocc-core/OCC fehlt im Kernel /Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/bin/python.
Fuer diesen Lauf wird nur der CAD-Adapter auf synthetic_json umgestellt.
Baue die Factory-Pipeline auf ...


/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(



Konfiguration:
  Antwortmodell:     llama3.2:3b
  Antwortstrategie:  multi_agent
  Review aktiv:      True
  Revisionsrunden:   0
  Top-K Chunks:      5
  Output-Format:     standard
  Laufzeitbudget:    30 Minuten
  Collection:        knowledge_base
  Testset:           /Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/test_verzahnung/rag_evaluation/rag_evaluation_testset.json


## 1. Fester CAD-Kontext

In [2]:
# Ein fester CAD-Kontext fuer alle Faelle und beide Notebooks.
if config.cad_adapter.implementation == "synthetic_json":
    cad_dateien = synthetic_cad_adapter.list_files()
    if not cad_dateien:
        raise FileNotFoundError("Keine synthetischen CAD-Testdaten vorhanden.")
    cad_metadata = synthetic_cad_adapter.load_file(cad_dateien[0])
    cad_quelle = f"synthetisch: {cad_dateien[0].name}"
else:
    cad_metadata = cad_adapter.extract(file_path=STEP_FILE_PATH)
    cad_quelle = f"echte STEP-Datei: {STEP_FILE_PATH.name}"

print(f"CAD-Kontext: {cad_quelle}")
print(json.dumps(cad_metadata, ensure_ascii=False, indent=2))


CAD-Kontext: synthetisch: gear_01.json
{
  "schema_version": "1.0",
  "source_file": "gear_01.step",
  "gear_type": "spur",
  "confidence": 0.95,
  "basic_geometry": {
    "outer_diameter_mm": 44.0,
    "root_diameter_mm": 35.0,
    "pitch_diameter_mm": 40.0,
    "face_width_mm": 20.0,
    "total_width_mm": 20.0,
    "hub_bore_diameter_mm": 10.0,
    "volume_mm3": 28839.8,
    "surface_area_mm2": 5805.7
  },
  "tooth_profile": {
    "num_teeth": 20,
    "module_mm": 2.0,
    "helix_angle_deg": 0.0,
    "pressure_angle_deg": 20.0,
    "tooth_height_mm": 4.5,
    "addendum_mm": 2.0,
    "dedendum_mm": 2.5,
    "profile_shift_x": 0.0,
    "root_fillet_radius_mm": 0.76,
    "tooth_thickness_mm": 3.142
  },
  "topology": {
    "is_internal_gear": false,
    "symmetry_type": "rotational",
    "cone_angle_deg": null,
    "shaft_angle_deg": null,
    "worm_starts": null,
    "keyway_present": true,
    "has_flanges": false
  },
  "material_context": {
    "material": "16MnCr5",
    "mass_kg": 

## 2. Gespeichertes Frage-Chunk-Testset laden

In [3]:
# Das Testset wird ausschliesslich geladen (Erzeugung erfolgt einmalig in create_testset.py).
testset = lade_testset(TESTSET_PATH)
if testset["collection_name"] != config.vector_store.collection_name:
    raise ValueError(
        "Das gespeicherte Testset gehoert zu einer anderen Collection als die aktive Pipeline. "
        "Bitte create_testset.py fuer die aktuelle Wissensbasis erneut ausfuehren."
    )

testfaelle = testset["items"] if MAX_CASES is None else testset["items"][:MAX_CASES]
print(f"Testset geladen: {TESTSET_PATH}")
print(f"Generator-Modell: {testset['generator_model']} | Faelle: {len(testfaelle)} / {testset['n_items']}")
print(f"Zufallsbasis: {testset['qdrant_chunks_total']} Chunks aus der gesamten Collection")
for item in testfaelle:
    print(f"  {item['id']}: {item['source_name']} S.{item['page_number']} -> {item['question']}")


Testset geladen: /Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/test_verzahnung/rag_evaluation/rag_evaluation_testset.json
Generator-Modell: llama3.2:3b | Faelle: 20 / 20
Zufallsbasis: 481 Chunks aus der gesamten Collection
  rag_eval_001: gear_skiving_technology.pdf S.6 -> Wie beeinflusst die Rakekante des Schneidbogens auf das Verhalten der Schneidegeometrie und wie kann diese Beziehung mathematisch modelliert werden?
  rag_eval_002: LCS 600-1200.pdf S.2 -> Wie viele Drehzahlen bis 12.000 min-1 bietet der Schleifkopf einer LCS-Baureihe-Maschine zur Verfügung?
  rag_eval_003: DIN ISO 21771_2014-08-00_DE_2144663.pdf S.81 -> Was ist die Formel zur Berechnung der minimal zulässigen Zahndickensehne, die den Unterschied zwischen maximaler und minimaler Sehnenlängen berücksichtigt?
  rag_eval_004: ISO-TR 10825-2_2022-10-00_EN_3392173.pdf S.40 -> Was ist die Ursache für Tooth-Shear-Fraß in Gears, und wie wird er durch eine einzelne extreme Belastung verursacht?
  rag_eval_0

## 3. End-to-End-Durchlauf ueber das gesamte Testset

In [4]:
ergebnisse = []

for index, fall in enumerate(testfaelle, start=1):
    if not restbudget_ok(NOTEBOOK_START, MAX_RUNTIME_SECONDS, reserve_seconds=190):
        print("Laufzeitbudget erreicht; verbleibende Faelle werden nicht mehr gestartet.")
        break

    eintrag = {
        **fall,
        "treffer_anzahl": 0,
        "answer_text": None,
        "score_mean": None,
        "score_max": None,
        "status": "gestartet",
        "_treffer_texte": [],
    }
    try:
        treffer = rufe_retriever_auf(retriever, fall["question"], cad_metadata)[:TOP_K]
        eintrag["treffer_anzahl"] = len(treffer)
        eintrag["_treffer_texte"] = [rc.chunk.text for rc in treffer]
        if not treffer:
            eintrag["status"] = "keine Treffer"
            ergebnisse.append(eintrag)
            continue

        antwort = generiere_antwort(
            answer_gen,
            frage=fall["question"],
            treffer=treffer,
            cad_metadata=cad_metadata,
            output_format=OUTPUT_FORMAT,
        )
        answer_text = str(antwort.get("answer_text") or "").strip()
        if not answer_text:
            eintrag["status"] = "leere Antwort"
            ergebnisse.append(eintrag)
            continue

        eintrag["answer_text"] = answer_text
        eintrag["status"] = "gueltig"
    except Exception as exc:
        eintrag["status"] = f"Fehler: {type(exc).__name__}: {exc}"
    ergebnisse.append(eintrag)
    print(f"[{index}/{len(testfaelle)}] {eintrag['status']}", flush=True)

# Scoring: Antwort-Embedding gegen die Top-K-Chunk-Embeddings.
#   score_mean = mittlere Kosinus-Aehnlichkeit ueber die Top-K-Chunks
#   score_max  = maximale Kosinus-Aehnlichkeit ueber die Top-K-Chunks
gueltige = [e for e in ergebnisse if e["answer_text"] and e["_treffer_texte"]]
if gueltige:
    antwort_vecs = embedder.embed([e["answer_text"] for e in gueltige]).dense_vectors
    alle_treffer_texte = [text for e in gueltige for text in e["_treffer_texte"]]
    alle_treffer_vecs = embedder.embed(alle_treffer_texte).dense_vectors

    offset = 0
    for eintrag, antwort_vec in zip(gueltige, antwort_vecs):
        anzahl = len(eintrag["_treffer_texte"])
        chunk_vecs = alle_treffer_vecs[offset: offset + anzahl]
        offset += anzahl
        sims = [kosinus_aehnlichkeit(antwort_vec, vec) for vec in chunk_vecs]
        eintrag["score_mean"] = sum(sims) / len(sims)
        eintrag["score_max"] = max(sims)

laufzeit_seconds = time.monotonic() - NOTEBOOK_START
print(f"Abgeschlossen: {len(ergebnisse)}/{len(testfaelle)} Faelle in {laufzeit_seconds / 60:.2f} Minuten.")


multiagent_generate_failed; Fallback auf Single-Pass
Traceback (most recent call last):
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/httpx/_transports/default.py", line 250, in handle_request
    resp = self._pool.handle_request(req)
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/httpcore/_sync/connection_pool.py", line 256, in handle_request
    raise exc from None
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/httpcore/_sync/connection_pool.py", line 236, in handle_request
    response = connection.handle_request(
  File "/Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/.venv/lib/python3.9/site-packages/http

[1/20] gueltig
[2/20] gueltig
[3/20] gueltig
[4/20] gueltig
[5/20] gueltig
[6/20] gueltig
[7/20] gueltig
[8/20] gueltig
[9/20] gueltig
[10/20] gueltig
[11/20] gueltig
Laufzeitbudget erreicht; verbleibende Faelle werden nicht mehr gestartet.
Abgeschlossen: 11/20 Faelle in 30.70 Minuten.


## 4. Auswertung

In [5]:
def zeige_ergebnistabelle(ergebnisse, score_spalten):
    kopf = ["Fall", "Quelle / Seite", "Frage", "Treffer", *score_spalten, "Antwort", "Status"]
    kopf_html = "".join(f"<th>{spalte}</th>" for spalte in kopf)
    zeilen = []
    for eintrag in ergebnisse:
        scores = [
            "\u2014" if eintrag.get(spalte) is None else f"{eintrag[spalte]:.4f}"
            for spalte in score_spalten
        ]
        werte = [
            eintrag["id"],
            f"{eintrag['source_name']} / S.{eintrag['page_number']}",
            kuerzen(eintrag["question"], 110),
            eintrag["treffer_anzahl"],
            *scores,
            kuerzen(eintrag.get("answer_text"), 130),
            eintrag.get("status", ""),
        ]
        zellen = "".join(f"<td>{str(wert)}</td>" for wert in werte)
        zeilen.append(f"<tr>{zellen}</tr>")
    display(HTML(
        f'<div style="max-height:650px;overflow:auto;border:1px solid #999;border-radius:5px;">'
        f'<table style="border-collapse:collapse;width:100%;font-size:.84em;">'
        f'<thead style="position:sticky;top:0;background:#e2e2e2;"><tr>{kopf_html}</tr></thead>'
        f'<tbody>{"".join(zeilen)}</tbody></table></div>'
        '<style>th,td{padding:6px 8px;border-bottom:1px solid #ddd;text-align:left;vertical-align:top}'
        'tbody tr:nth-child(even){background:#f5f5f5}</style>'
    ))


def aggregat(werte):
    werte = [wert for wert in werte if wert is not None]
    if not werte:
        return {"n": 0, "mean": None, "median": None, "min": None, "max": None, "std": None}
    return {
        "n": len(werte),
        "mean": statistics.mean(werte),
        "median": statistics.median(werte),
        "min": min(werte),
        "max": max(werte),
        "std": statistics.pstdev(werte),
    }


In [6]:
zeige_ergebnistabelle(ergebnisse, ["score_mean", "score_max"])

agg_mean = aggregat([e["score_mean"] for e in ergebnisse])
agg_max = aggregat([e["score_max"] for e in ergebnisse])
print("score_mean (Mittel ueber Top-K):", agg_mean)
print("score_max  (Max ueber Top-K):   ", agg_max)
print()
print(f"DURCHSCHNITT score_mean ueber alle Faelle: {agg_mean['mean']}")
print(f"DURCHSCHNITT score_max  ueber alle Faelle: {agg_max['mean']}")


Fall,Quelle / Seite,Frage,Treffer,score_mean,score_max,Antwort,Status
rag_eval_001,gear_skiving_technology.pdf / S.6,Wie beeinflusst die Rakekante des Schneidbogens auf das Verhalten der Schneidegeometrie und wie kann diese Be…,5,0.6466,0.6890,Die Rakekante des Schneidbogens spielt eine entscheidende Rolle bei der Bestimmung der Schneidegeometrie und beeinflusst das Verh…,gueltig
rag_eval_002,LCS 600-1200.pdf / S.2,Wie viele Drehzahlen bis 12.000 min-1 bietet der Schleifkopf einer LCS-Baureihe-Maschine zur Verfügung?,5,0.6120,0.6845,"Die LCS-Baureihe-Maschinen bieten einen Schleifkopf zur Verfügung, der Drehzahlen bis zu 12.000 min-1 anbietet.",gueltig
rag_eval_003,DIN ISO 21771_2014-08-00_DE_2144663.pdf / S.81,"Was ist die Formel zur Berechnung der minimal zulässigen Zahndickensehne, die den Unterschied zwischen maxima…",5,0.6960,0.7243,"Die Formel zur Berechnung der minimal zulässigen Zahndickensehne, die den Unterschied zwischen maximaler und minimaler Sehnenläng…",gueltig
rag_eval_004,ISO-TR 10825-2_2022-10-00_EN_3392173.pdf / S.40,"Was ist die Ursache für Tooth-Shear-Fraß in Gears, und wie wird er durch eine einzelne extreme Belastung veru…",5,0.6614,0.6825,"Tooth-Shear-Fraß in Gears wird durch eine einzelne extreme Belastung verursacht, die zu einer plötzlichen und starken Deformation…",gueltig
rag_eval_005,ISO 10825_1995-08-00_ML_2839045.pdf / S.33,Wie wird die Plastizdeformation bei der Bearbeitung von Zähnen durch den Überlastungsgrad und das Frotten bee…,5,0.6532,0.6937,Plastizdeformation bei der Bearbeitung von Zähnen wird durch den Überlastungsgrad und das Frotten stark beeinflusst. Wenn die Zäh…,gueltig
rag_eval_006,ISO-TR 10825-2_2022-10-00_EN_3392173.pdf / S.51,"Was ist die Ursache für das Erschöpfen von Flüssigkeiten in der Nähe von Zahnreifen, was zu einer lokalen Dru…",5,0.6262,0.6692,"Das Erschöpfen von Flüssigkeiten in der Nähe von Zahnreifen ist eine Folge des Cavitation. Diese Vorgang tritt auf, wenn die Druc…",gueltig
rag_eval_007,gear_skiving_technology.pdf / S.12,Wie beeinflusst die Verwendung von skiving-Techniken bei der Bearbeitung von helical Gears die Rake- und Schn…,5,0.6951,0.7095,Die Verwendung von skiving-Techniken bei der Bearbeitung von helical Gears kann die Rake- und Schneidewinkel des Werkzeuges erheb…,gueltig
rag_eval_008,DIN ISO 1328 ISO-Toleranzsystem pt.2.pdf / S.29,Wie werden in der DIN ISO 1328-2:2021-09 flankenverwandte Abweichungen von Zahnflächen klassifiziert und wie …,5,0.7157,0.7583,Die Abweichungen von Zahnflächen werden in der DIN ISO 1328-2:2021-09 flankenverwandt klassifiziert und unterteilt in verschieden…,gueltig
rag_eval_009,DIN ISO 1328 ISO-Toleranzsystem.pdf / S.25,"Was ist die Bezeichnung für den Abstand zwischen zwei Kopien der Sollflankenlinie, der über dem Flankenlinien…",5,0.6386,0.6784,"Der Abstand zwischen zwei Kopien der Sollflankenlinie, der über dem Flankenlinien-Auswertebereich einschließt, wird als Flankenli…",gueltig
rag_eval_010,gear_skiving_technology.pdf / S.16,Wie können künstliche Intelligenz-Technologien zur Vorhersage der Lebensdauer von Werkzeugen und zum Überwach…,5,0.6487,0.6795,**Optimierung von künstlicher Intelligenz-Technologien zur Vorhersage der Lebensdauer von Werkzeugen und zum Überwachung von Vers…,gueltig


score_mean (Mittel ueber Top-K): {'n': 11, 'mean': 0.6644545532113663, 'median': 0.6531706635682669, 'min': 0.6119526668186304, 'max': 0.715670920742676, 'std': 0.03406353587463037}
score_max  (Max ueber Top-K):    {'n': 11, 'mean': 0.707736992635604, 'median': 0.689035250179256, 'min': 0.6691979661423799, 'max': 0.8162310989174667, 'std': 0.042068410818856525}

DURCHSCHNITT score_mean ueber alle Faelle: 0.6644545532113663
DURCHSCHNITT score_max  ueber alle Faelle: 0.707736992635604


## 5. JSON-Log

In [7]:
LOG_DIR.mkdir(parents=True, exist_ok=True)
zeitstempel = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
log_pfad = LOG_DIR / f"{zeitstempel}_llm_eval.json"

log_daten = {
    "zeitstempel": zeitstempel,
    "testset_path": str(TESTSET_PATH),
    "testset_created_at": testset["created_at"],
    "collection_name": config.vector_store.collection_name,
    "antwortmodell": config.answer_generator.model_name,
    "antwortstrategie": config.answer_generator.implementation,
    "output_format": OUTPUT_FORMAT,
    "top_k": TOP_K,
    "cad_quelle": cad_quelle,
    "runtime_seconds": laufzeit_seconds,
    "n_faelle": len(ergebnisse),
    "aggregat": {"score_mean": agg_mean, "score_max": agg_max},
    "eintraege": [
        {k: v for k, v in eintrag.items() if not k.startswith("_") and k != "embedding"}
        for eintrag in ergebnisse
    ],
}
log_pfad.write_text(json.dumps(log_daten, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Log gespeichert: {log_pfad}")


Log gespeichert: /Users/maxhammerstein/Projects/KI-Copilot für Verzahnungswissen/test_verzahnung/rag_evaluation/logs/20260630_144747_llm_eval.json
